In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp03/ex_3.py ──
"""
Shared infrastructure for MLFP03 Exercise 3 — The Classical ML Zoo.

Contains: e-commerce churn data loading, preprocessing, CV strategy,
2D PCA projection for decision boundary plots, model comparison helpers,
and a shared ModelVisualizer-backed plot utility.

Technique-specific code (model fitting, parameter sweeps, from-scratch
Gini, OOB convergence, decision guide) does NOT belong here — it lives
in the per-technique files under modules/mlfp03/solutions/ex_3/.
"""

import time
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score

from kailash_ml import ModelVisualizer, PreprocessingPipeline
from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT
# ════════════════════════════════════════════════════════════════════════

setup_environment()
np.random.seed(42)

# Output directory for comparison artifacts (HTML plots, tables)
OUTPUT_DIR = Path("outputs") / "ex3_model_zoo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# E-commerce churn dataset — Singapore APAC retail churn scenario
DATASET_MODULE = "mlfp03"
DATASET_FILE = "ecommerce_customers.parquet"
TARGET_COL = "churned"

# SVM with RBF kernel is O(n²) — cap the training set so every technique
# in the zoo fits in a few seconds on a laptop.
SUBSAMPLE_N = 5000
RANDOM_SEED = 42

# Drop columns that are text/ID or leak the target
DROP_COLS = ["customer_id", "review_text", "product_categories"]


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING + PREPROCESSING
# ════════════════════════════════════════════════════════════════════════


def load_ecommerce_churn() -> pl.DataFrame:
    """Load the Singapore e-commerce churn dataset (polars DataFrame).

    Drops text/ID columns and subsamples for SVM tractability.
    """
    loader = MLFPDataLoader()
    df = loader.load(DATASET_MODULE, DATASET_FILE)
    df = df.sample(n=min(SUBSAMPLE_N, df.height), seed=RANDOM_SEED)
    keep = [c for c in df.columns if c not in DROP_COLS]
    return df.select(keep)


def build_train_test_split() -> dict[str, Any]:
    """Return a fully prepared dict: X_train, X_test, y_train, y_test, feature_names, cv.

    Uses kailash_ml PreprocessingPipeline with z-score normalisation and
    ordinal categorical encoding. Every technique file calls this so all
    models share identical folds and identical preprocessing.
    """
    df = load_ecommerce_churn()

    pipeline = PreprocessingPipeline()
    result = pipeline.setup(
        data=df,
        target=TARGET_COL,
        train_size=0.8,
        seed=RANDOM_SEED,
        normalize=True,
        normalize_method="zscore",
        categorical_encoding="ordinal",
        imputation_strategy="median",
    )

    feature_cols = [c for c in result.train_data.columns if c != TARGET_COL]
    X_train, y_train, col_info = to_sklearn_input(
        result.train_data,
        feature_columns=feature_cols,
        target_column=TARGET_COL,
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_cols,
        target_column=TARGET_COL,
    )
    feature_names = col_info["feature_columns"]

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

    return {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "feature_names": feature_names,
        "cv": cv,
        "churn_rate": float(np.mean(y_train)),
    }


# ════════════════════════════════════════════════════════════════════════
# 2D PCA PROJECTION — shared so every technique plots on the same axes
# ════════════════════════════════════════════════════════════════════════


def project_2d(X_train: np.ndarray, X_test: np.ndarray) -> dict[str, Any]:
    """Fit PCA(2) on X_train and project both train and test.

    Returns {X_train_2d, X_test_2d, explained_variance, pca}.
    """
    pca = PCA(n_components=2, random_state=RANDOM_SEED)
    X_train_2d = pca.fit_transform(X_train)
    X_test_2d = pca.transform(X_test)
    return {
        "X_train_2d": X_train_2d,
        "X_test_2d": X_test_2d,
        "explained_variance": pca.explained_variance_ratio_,
        "pca": pca,
    }


# ════════════════════════════════════════════════════════════════════════
# CROSS-VALIDATION HELPER — keep every parameter sweep one line
# ════════════════════════════════════════════════════════════════════════


def cv_accuracy_f1(
    estimator: Any,
    X: np.ndarray,
    y: np.ndarray,
    cv: Any,
) -> tuple[float, float]:
    """Return (mean_accuracy, mean_f1) for a 5-fold CV."""
    acc = cross_val_score(estimator, X, y, cv=cv, scoring="accuracy").mean()
    f1 = cross_val_score(estimator, X, y, cv=cv, scoring="f1").mean()
    return float(acc), float(f1)


# ════════════════════════════════════════════════════════════════════════
# EVALUATION — train on full set, measure timing, return pred/prob/metrics
# ════════════════════════════════════════════════════════════════════════


def fit_and_evaluate(
    estimator: Any,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    name: str,
) -> dict[str, Any]:
    """Fit, predict, score, and time a single model.

    Returns a dict with keys: name, model, pred, prob, train_time,
    accuracy, f1, auc_roc.
    """
    t0 = time.perf_counter()
    estimator.fit(X_train, y_train)
    train_time = time.perf_counter() - t0

    pred = estimator.predict(X_test)
    if hasattr(estimator, "predict_proba"):
        prob = estimator.predict_proba(X_test)[:, 1]
    else:
        # Decision-function fallback (never used by the zoo but keeps contract)
        prob = estimator.decision_function(X_test)

    return {
        "name": name,
        "model": estimator,
        "pred": pred,
        "prob": prob,
        "train_time": float(train_time),
        "accuracy": float(accuracy_score(y_test, pred)),
        "f1": float(f1_score(y_test, pred)),
        "auc_roc": float(roc_auc_score(y_test, prob)),
    }


def print_classification_report(y_test: np.ndarray, pred: np.ndarray) -> None:
    """Print sklearn classification report with churn-friendly target names."""
    print(
        classification_report(
            y_test,
            pred,
            target_names=["Retained", "Churned"],
        )
    )


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION — Plotly via kailash_ml.ModelVisualizer
# ════════════════════════════════════════════════════════════════════════


def get_visualizer() -> ModelVisualizer:
    """Return a ModelVisualizer instance (polars-native plots)."""
    return ModelVisualizer()


def save_metric_comparison(
    metric_dict: dict[str, dict[str, float]], fname: str
) -> Path:
    """Save a metric_comparison plot to OUTPUT_DIR/fname and return the path."""
    viz = get_visualizer()
    fig = viz.metric_comparison(metric_dict)
    fig.update_layout(title="Classical ML Zoo — Performance Comparison")
    out = OUTPUT_DIR / fname
    fig.write_html(str(out))
    return out


# ════════════════════════════════════════════════════════════════════════
# DECISION BOUNDARY MESH — shared helper so every technique file uses
# the same axes, grid, and figure style.
# ════════════════════════════════════════════════════════════════════════


def decision_boundary_mesh(
    X_2d: np.ndarray,
    step: float = 0.1,
    pad: float = 0.5,
) -> tuple[np.ndarray, np.ndarray]:
    """Return (xx, yy) meshgrid covering the 2D PCA projection."""
    x_min, x_max = X_2d[:, 0].min() - pad, X_2d[:, 0].max() + pad
    y_min, y_max = X_2d[:, 1].min() - pad, X_2d[:, 1].max() + pad
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, step),
        np.arange(y_min, y_max, step),
    )
    return xx, yy


# ════════════════════════════════════════════════════════════════════════
# SINGAPORE E-COMMERCE CHURN — business-impact constants
# ════════════════════════════════════════════════════════════════════════
# Public industry figures used for the "Apply" phases. Sources in reading
# notes (SGX retail analyst reports, Shopee/Lazada 2024 ops reviews).

AVG_CUSTOMER_LIFETIME_VALUE_SGD = 420.0  # avg 12-month CLV per retained SG customer
RETENTION_OFFER_COST_SGD = 18.0  # targeted promo cost per flagged customer
MONTHLY_ACTIVE_CUSTOMERS = 250_000  # typical mid-market SG e-commerce platform
ANNUAL_CHURN_BASELINE = 0.22  # industry baseline annual churn


def churn_saved_dollars(true_positives: int) -> float:
    """Dollar value of correctly identified churners (retention offer accepted).

    Assumes a 40% offer-acceptance rate and the retained lifetime value
    net of offer cost. Public industry benchmarks — not proprietary data.
    """
    accept_rate = 0.40
    net_value_per_save = AVG_CUSTOMER_LIFETIME_VALUE_SGD - RETENTION_OFFER_COST_SGD
    return round(true_positives * accept_rate * net_value_per_save, 2)




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 3.4: Decision Trees (Gini from scratch + sklearn)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Compute Gini impurity and entropy from scratch in numpy
#   - Simulate the first split of a decision tree by exhaustive search
#   - Verify the from-scratch split matches sklearn's first split
#   - Tune max_depth with cross-validation
#   - Read a tree's feature-importance ranking as an audit artefact
#   - Use decision trees for interpretable Singapore compliance scenarios
#
# PREREQUISITES: 01_svm.py (shared preprocessing), MLFP01 numpy basics
#
# ESTIMATED TIME: ~35 min
#
# TASKS:
#   1. Theory — Gini impurity, entropy, information gain
#   2. Build — from-scratch Gini + best-split search
#   3. Train — sklearn DecisionTreeClassifier with depth tuning
#   4. Visualise — tree structure, feature importance, 2D boundary
#   5. Apply — interpretable churn flagging for PDPA audit
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
from dotenv import load_dotenv
from sklearn.tree import DecisionTreeClassifier, export_text


load_dotenv()



## THEORY — Impurity, Entropy, and Greedy Splitting


In [ ]:
# A decision tree partitions the feature space by repeatedly asking
# "which feature, at which threshold, most reduces class impurity?"
#
# Gini impurity (the default sklearn split criterion):
#     G(node) = 1 - Σ p_k²
#         Pure node (all one class): G = 0
#         Binary 50/50:              G = 0.5  (maximum)
#
# Entropy (ID3/C4.5 criterion):
#     H(node) = - Σ p_k log₂(p_k)
#
# Information gain for a candidate split s that partitions node N into
# children C_1, ..., C_m:
#     IG(s) = impurity(N) - Σ_j (|C_j| / |N|) * impurity(C_j)
#
# The tree is GREEDY: at every node it picks the locally best split,
# without backtracking or global optimisation. This is why an ensemble
# of de-correlated trees (Random Forest, 05_random_forest.py) usually
# beats a single tree.



## TASK 2 — BUILD: from-scratch Gini + best-split search


Gini impurity: G = 1 - Σ p_k² for k classes.


Shannon entropy: H = - Σ p_k log₂(p_k).


For each feature, try midpoints between consecutive unique values as
    candidate thresholds, compute the weighted-child Gini, and track the
    split with the largest reduction in parent impurity. Samples at most
    50 thresholds per feature to keep the search tractable.


In [ ]:
print("\n" + "=" * 70)
print("  MLFP03 Exercise 3.4 — Decision Trees")
print("=" * 70)


def gini_impurity(y: np.ndarray) -> float:
    _, counts = np.unique(y, return_counts=True)
    proportions = counts / len(y)
    return float(1.0 - np.sum(proportions**2))


def entropy(y: np.ndarray) -> float:
    _, counts = np.unique(y, return_counts=True)
    proportions = counts / len(y)
    proportions = proportions[proportions > 0]
    return float(-np.sum(proportions * np.log2(proportions)))


# Gini for canonical distributions
print("\n--- Gini / Entropy for reference distributions ---")
print(f"{'Distribution':<25} {'Gini':>8} {'Entropy':>8}")
print("-" * 43)
for label, y_demo in [
    ("Pure: [100, 0]", np.array([0] * 100)),
    ("50/50: [50, 50]", np.array([0] * 50 + [1] * 50)),
    ("90/10: [90, 10]", np.array([0] * 90 + [1] * 10)),
    ("70/30: [70, 30]", np.array([0] * 70 + [1] * 30)),
    ("3-class [40,30,30]", np.array([0] * 40 + [1] * 30 + [2] * 30)),
]:
    print(f"  {label:<23} {gini_impurity(y_demo):>8.4f} {entropy(y_demo):>8.4f}")


def best_split_search(
    X: np.ndarray,
    y: np.ndarray,
) -> tuple[int, float, float]:
    n = len(y)
    parent_gini = gini_impurity(y)
    best_gain = 0.0
    best_feature_idx = 0
    best_threshold = 0.0

    rng = np.random.default_rng(RANDOM_SEED)
    for feat_idx in range(X.shape[1]):
        sorted_vals = np.unique(X[:, feat_idx])
        if len(sorted_vals) < 2:
            continue
        thresholds = (sorted_vals[:-1] + sorted_vals[1:]) / 2
        if len(thresholds) > 50:
            thresholds = rng.choice(thresholds, 50, replace=False)
        for threshold in thresholds:
            left_mask = X[:, feat_idx] <= threshold
            n_left = int(left_mask.sum())
            n_right = n - n_left
            if n_left == 0 or n_right == 0:
                continue
            g_left = gini_impurity(y[left_mask])
            g_right = gini_impurity(y[~left_mask])
            weighted = (n_left / n) * g_left + (n_right / n) * g_right
            gain = parent_gini - weighted
            if gain > best_gain:
                best_gain = gain
                best_feature_idx = feat_idx
                best_threshold = float(threshold)
    return best_feature_idx, best_threshold, float(best_gain)


data = build_train_test_split()
X_train, X_test = data["X_train"], data["X_test"]
y_train, y_test = data["y_train"], data["y_test"]
cv = data["cv"]
feature_names = data["feature_names"]

print(f"\nTrain: {X_train.shape}, Test: {X_test.shape}")
print("\n--- From-scratch best-split search on training data ---")
best_feat_idx, best_threshold, best_gain = best_split_search(X_train, y_train)
best_feat_name = feature_names[best_feat_idx]
print(
    f"  Feature: {best_feat_name}\n"
    f"  Threshold: {best_threshold:.4f}\n"
    f"  Gini gain: {best_gain:.4f}"
)

# Verify against sklearn's depth-1 stump
stump = DecisionTreeClassifier(max_depth=1, random_state=RANDOM_SEED)
stump.fit(X_train, y_train)
sklearn_feat = feature_names[int(stump.tree_.feature[0])]
sklearn_thresh = float(stump.tree_.threshold[0])
print(f"\nsklearn stump  : feature={sklearn_feat}, threshold={sklearn_thresh:.4f}")
print(
    "  Match"
    if sklearn_feat == best_feat_name
    else "  Close (search samples thresholds; sklearn uses all)"
)



### Checkpoint 1


In [ ]:
assert best_gain > 0, "Best split should have positive Gini gain"
assert abs(gini_impurity(np.array([0] * 100))) < 1e-9, "Pure Gini should be 0"
assert abs(gini_impurity(np.array([0] * 50 + [1] * 50)) - 0.5) < 1e-9, "50/50 = 0.5"
print("\n[ok] Checkpoint 1 passed — Gini + best-split verified\n")



## TASK 3 — TRAIN: sklearn DecisionTreeClassifier with depth tuning


In [ ]:
DEPTHS = [2, 3, 5, 7, 10, 15, None]
print("--- max_depth sweep ---")
print(f"{'depth':>8} {'CV Accuracy':>14} {'CV F1':>10}")
print("-" * 36)
depth_results: dict[str, dict[str, float | int | None]] = {}
for depth in DEPTHS:
    acc, f1 = cv_accuracy_f1(
        DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_SEED),
        X_train,
        y_train,
        cv,
    )
    label = str(depth) if depth is not None else "None"
    depth_results[label] = {"depth": depth, "accuracy": acc, "f1": f1}
    print(f"{label:>8} {acc:>14.4f} {f1:>10.4f}")

best_depth_label = max(depth_results, key=lambda d: depth_results[d]["f1"])  # type: ignore[arg-type]
best_depth = depth_results[best_depth_label]["depth"]
print(f"\nBest depth: {best_depth_label}")

dt_result = fit_and_evaluate(
    DecisionTreeClassifier(max_depth=best_depth, random_state=RANDOM_SEED),
    X_train,
    y_train,
    X_test,
    y_test,
    name=f"DecisionTree (depth={best_depth_label})",
)
dt_model = dt_result["model"]

print(
    f"\n{dt_result['name']}: trained in {dt_result['train_time']:.4f}s | "
    f"accuracy={dt_result['accuracy']:.4f} | "
    f"F1={dt_result['f1']:.4f} | AUC={dt_result['auc_roc']:.4f}"
)
print_classification_report(y_test, dt_result["pred"])
print(f"Tree depth: {dt_model.get_depth()}, leaves: {dt_model.get_n_leaves()}")



## TASK 4 — VISUALISE: tree structure, feature importance, 2D boundary


In [ ]:
print("\n--- Tree structure (first 3 levels) ---")
print(export_text(dt_model, feature_names=feature_names, max_depth=3)[:1500])

importances = sorted(
    zip(feature_names, dt_model.feature_importances_),
    key=lambda pair: pair[1],
    reverse=True,
)
print("\n--- Top feature importances ---")
print(f"{'feature':<30} {'importance':>12}")
print("-" * 44)
for name, imp in importances[:10]:
    bar = "#" * int(imp * 50)
    print(f"{name:<30} {imp:>12.4f}  {bar}")

# 2D decision boundary
pca_bundle = project_2d(X_train, X_test)
X_train_2d = pca_bundle["X_train_2d"]
dt_2d = DecisionTreeClassifier(max_depth=best_depth, random_state=RANDOM_SEED)
dt_2d.fit(X_train_2d, y_train)
xx, yy = decision_boundary_mesh(X_train_2d)
Z = dt_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

viz = get_visualizer()
imp_plot = {"importance": [imp for _, imp in importances[:10]]}
fig = viz.training_history(imp_plot, x_label="feature rank (top 10)")
fig.update_layout(title="Decision Tree — top 10 feature importances")
out = OUTPUT_DIR / "ex3_04_tree_importance.html"
fig.write_html(str(out))
print(f"\nSaved: {out}")
print(
    f"Decision mesh shape: {Z.shape} | "
    f"PCA variance captured: {pca_bundle['explained_variance'].sum():.2%}"
)
print("Decision-tree boundaries are AXIS-ALIGNED rectangles — no diagonal lines.")



### Checkpoint 2


In [ ]:
assert dt_result["accuracy"] > 0.5, "Decision tree must beat random"
assert Z.shape[0] > 0 and Z.shape[1] > 0, "Decision boundary mesh is empty"
print("[ok] Checkpoint 2 passed — tree trained and visualised\n")



## TASK 5 — APPLY: interpretable churn flagging for PDPA audit


In [ ]:
# SCENARIO: Singapore's PDPA and MAS guidance on automated decisioning
# require a firm to explain any customer-impacting model output. A
# marketing retention model that recommends a S$18 offer is low-risk,
# but a credit or lending use case is not. Regulators accept decision
# trees as inherently interpretable — every prediction walks a sequence
# of feature-threshold rules that a human can read.
#
# Why a decision tree fits:
#   - Every prediction is a path: "recency > 35 days AND cart
#     abandonments > 3 AND support tickets > 0 -> flag for retention".
#   - Audit tool output is the same as the model itself.
#   - Feature importance gives a single-column risk summary.
#
# LIMITATIONS:
#   - High variance: a small change in training data can flip the top
#     split and produce a dramatically different tree.
#   - Depth-tuned trees still underperform ensembles on raw accuracy.
#   - The single-tree recall is capped by axis-aligned splits —
#     diagonal decision surfaces need many steps.

true_positives = int(((dt_result["pred"] == 1) & (y_test == 1)).sum())
dollars_saved = churn_saved_dollars(true_positives)
print(f"\nBusiness impact on held-out test set ({len(y_test)} customers):")
print(f"  True positives (churners caught): {true_positives}")
print(f"  Net retention value at 40% offer acceptance: S${dollars_saved:,.2f}")



## REFLECTION


[x] Gini impurity and entropy computed from scratch
  [x] Best-split exhaustive search, verified against sklearn
  [x] max_depth tuned via CV
  [x] Held-out accuracy: {dt_result['accuracy']:.4f}, F1: {dt_result['f1']:.4f}
  [x] Tree feature importance as an audit artefact
  [x] Axis-aligned 2D decision boundary
  [x] Interpretability / compliance business case
      — S${dollars_saved:,.0f} retained on the held-out test fold

  Next: 05_random_forest.py — bag many de-correlated trees for OOB.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

